# Schema on read: loading a JSON file into a table with the right schema

**Goal:** read raw flight data from a JSON file and save it into a table.

Before saving, we check whether the **DataFrame's schema** matches the **target table's schema**:
- If they match, we simply save the data into the table.
- If there are mismatches (column order, data types), we fix them first.

This step is necessary to avoid write errors, and to avoid data landing in the wrong fields.

### Steps
1. Read the file and let Spark **infer** the schema
2. Compare it with the table and list the problems
3. Define the schema **ourselves** and read the file again with it (schema on read)
4. Check the data
5. Save it into the table

## The target table's structure

The DataFrame must end up with exactly these columns, **in this order**, with these types:

```sql
FL_DATE            DATE,
OP_CARRIER         STRING,
OP_CARRIER_FL_NUM  STRING,
ORIGIN             STRING,
ORIGIN_CITY_NAME   STRING,
DEST               STRING,
DEST_CITY_NAME     STRING,
CRS_DEP_TIME       LONG,
DEP_TIME           LONG,
WHEELS_ON          INT,
TAXI_IN            INT,
CRS_ARR_TIME       LONG,
ARR_TIME           LONG,
CANCELLED          INT,
DISTANCE           INT
```

## Step 1. Read the file and let Spark infer the schema

Options used:
- `mode = FAILFAST`: throw an exception as soon as Spark meets a corrupted record
  (the default, `PERMISSIVE`, silently puts `null`s instead; `DROPMALFORMED` drops the bad rows)
- `dateFormat = M/d/yyyy`: the pattern of the dates in the file (for example `1/1/2000`).
  Spark's default date pattern is `yyyy-MM-dd`, so it needs this to turn the text into a `DATE`.

In [3]:
# Read the JSON file and let Spark infer the schema
flight_time_raw_df = (
    spark.read.format("json")
    .option("mode", "FAILFAST") # throws an exception when it meets corrupted records
    .option("dateFormat", "M/d/yyyy") # specify the date format to parse the date columns
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/flight-time.json")
)

# Only shows the inferred schema, no data is read
flight_time_raw_df.printSchema()

root
 |-- ARR_TIME: long (nullable = true)
 |-- CANCELLED: long (nullable = true)
 |-- CRS_ARR_TIME: long (nullable = true)
 |-- CRS_DEP_TIME: long (nullable = true)
 |-- DEP_TIME: long (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DISTANCE: long (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: long (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- TAXI_IN: long (nullable = true)
 |-- WHEELS_ON: long (nullable = true)



## Step 2. Compare with the table: the problems

1. **Wrong column order.** The JSON reader puts inferred columns in **alphabetical order** (`ARR_TIME`, `CANCELLED`, ...), not in the table's order.
2. **`FL_DATE` is a `string`**, but it is a `DATE` in the table.
   - The JSON reader **never infers dates**: JSON has only strings, numbers and booleans.
   - So the `dateFormat` option above had **no effect**: it is used only when a schema says a column is a `DATE`.
3. **Other type mismatches:**
   - `OP_CARRIER_FL_NUM` was inferred as `long`, but it is a `STRING` in the table.
   - `WHEELS_ON`, `TAXI_IN`, `CANCELLED` and `DISTANCE` were inferred as `long`, but they are `INT` in the table (JSON inference always picks `long` for whole numbers).

## Step 3. Define the schema ourselves: schema on read

Connectors (DataFrame readers) may or may not infer the schema of the data correctly. That is the problem with **automatic schema inference**.

The fix: define the schema **explicitly** and tell the connector (here the JSON reader) to use it,
so it reads the data with the right types and **doesn't infer anything**. This is called **schema on read**.

A schema is built with the classes in `pyspark.sql.types`:
- `StructType([...])`: the whole schema, a list of columns (fields)
- `StructField("name", Type())`: one column, its name and its type
- `StringType()`, `DateType()`, `LongType()`, `IntegerType()`, ...: the types (`IntegerType` = SQL `INT`)

The fields are listed **in the table's order**, which also fixes problem 1.

In [4]:
# To implement schema on read, import the types from pyspark.sql.types
# and define the schema with the StructType and StructField classes
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, LongType

# Same columns, order and types as the target table
flight_schema = StructType([
    StructField("FL_DATE", DateType()),
    StructField("OP_CARRIER", StringType()),
    StructField("OP_CARRIER_FL_NUM", StringType()),
    StructField("ORIGIN", StringType()),
    StructField("ORIGIN_CITY_NAME", StringType()),
    StructField("DEST", StringType()),
    StructField("DEST_CITY_NAME", StringType()),
    StructField("CRS_DEP_TIME", LongType()),
    StructField("DEP_TIME", LongType()),
    StructField("WHEELS_ON", IntegerType()),
    StructField("TAXI_IN", IntegerType()),
    StructField("CRS_ARR_TIME", LongType()),
    StructField("ARR_TIME", LongType()),
    StructField("CANCELLED", IntegerType()),
    StructField("DISTANCE", IntegerType())
])

### Read the file again, with the schema

Pass the schema to the reader with `.schema(flight_schema)`.

**The order matters:** `.schema()` and `.option()` configure the **reader**, so they must come **before** `.load()`.
`.load()` is the step that creates the DataFrame. After it, `.schema` is a DataFrame property that only *returns* the schema.

Now `dateFormat` is used: the schema says `FL_DATE` is a `DATE`, so Spark parses `1/1/2000` with the `M/d/yyyy` pattern.

In [18]:
# Read flight-time.json with the schema we defined (schema on read)
flight_time_raw_with_schema_df = (
    spark.read.format("json")
    .option("mode", "FAILFAST") # throws an exception when it meets corrupted records
    .option("dateFormat", "M/d/yyyy") # specify the date format to parse the date columns
    .schema(flight_schema)  # use our schema instead of inferring one (must come before load)
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/flight-time.json")
)

# Same order and types as the table, and FL_DATE is now a date
flight_time_raw_with_schema_df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- CRS_DEP_TIME: long (nullable = true)
 |-- DEP_TIME: long (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- CRS_ARR_TIME: long (nullable = true)
 |-- ARR_TIME: long (nullable = true)
 |-- CANCELLED: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)



### Why specify the schema on read for CSV and JSON files

- **Correct and stable types:** without a schema, CSV reads every column as `string` and JSON never reads dates, so you get the wrong types or the wrong column order. An explicit schema matches the target table and, with `FAILFAST`, rejects bad records instead of letting them into the wrong fields.
- **Faster reads:** with `inferSchema`, Spark scans the file once just to guess the types. A given schema skips that extra pass.

> Self-describing formats (Parquet, ORC, Avro, Delta) and Spark tables store their schema with the data, so they don't need this.

## Step 4. Check the data

`printSchema()` only shows the schema **we gave** Spark: it doesn't read any data, so it can't catch bad values.
`FAILFAST` and `dateFormat` take effect only when the data is **actually read**, by an action like `display()` or `show()`.
If a date doesn't match `M/d/yyyy`, this is where the error appears.

`limit(3)` keeps only 3 rows, so the check is quick.

In [19]:
# Action: read a few rows, which parses the dates and checks the records
# (outside Databricks, use flight_time_raw_with_schema_df.show(3) to see the rows)
display(flight_time_raw_with_schema_df.limit(3))

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
0,2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",1115,1113,1343,5,1400,1348,0,946
1,2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",1315,1311,1536,7,1559,1543,0,946
2,2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",1415,1414,1642,9,1721,1651,0,946


## Step 5. Save the DataFrame to the table `flight_time_raw`

Now the DataFrame's schema matches the table's, so we can save it.

- `.write`: the **DataFrameWriter**, the opposite of `spark.read`
- `.mode("overwrite")`: replace the table's data if the table already exists
  (other modes: `append` adds rows, `errorifexists` (default) fails, `ignore` does nothing)
- `.saveAsTable("catalog.schema.table")`: save the data as a table (creates it if it doesn't exist)

`saveAsTable` is an **action**: it runs the whole job (read the file, apply the schema, write the table).

In [20]:
# Write the DataFrame into the table, replacing any existing data
flight_time_raw_with_schema_df.write.mode("overwrite").saveAsTable("dev.spark_db.flight_time_raw")

In [21]:
%sql
-- Check the saved table: read back a few rows
select * from dev.spark_db.flight_time_raw limit 3

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
0,2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",1115,1113,1343,5,1400,1348,0,946
1,2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",1315,1311,1536,7,1559,1543,0,946
2,2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",1415,1414,1642,9,1721,1651,0,946
